## MedImageParse3D: A Foundation Model for 3D Biomedical Volumetric Segmentation

## Prerequisites

This notebook requires the following setup. If you haven't completed these steps, please refer to the Getting Started section in the main README, which includes:

1. Deploying required models
2. Installing the Healthcare AI Toolkit
3. Downloading sample data
4. Configuring your `.env` file

### Required for This Notebook

- **Model Endpoint(s)**: `MIP3D_MODEL_ENDPOINT`

## Introduction

3D volumetric imaging is fundamental to modern medical practice, providing detailed spatial information through modalities such as **CT, MRI, and PET** scanning. Extracting meaningful insights from these volumes presents unique challenges compared to 2D images:

- **Volumetric Segmentation:** Precisely delineating 3D anatomical structures, organs, and pathologies across all slices.
- **3D Detection:** Identifying and localizing specific structures like tumors or lesions in three dimensions.
- **Spatial Analysis:** Understanding the spatial relationships between anatomical structures within a volume.

**MedImageParse3D** extends the MedImageParse foundation model to handle full 3D volumetric inputs. Using a **BoltzFormer** transformer-based architecture, it performs prompt-based volumetric segmentation across diverse 3D imaging modalities including CT, MRI, and PET.

**MedImageParse3D** is a biomedical foundation model developed in collaboration with **Microsoft Research, Providence Genomics**, and the **Paul G. Allen School of Computer Science and Engineering** at the University of Washington. It is part of the **Microsoft healthcare AI models** initiative.


## 1. Setup and Imports


In [ ]:
import os
from healthcareai_toolkit.clients import MedImageParseClient
from healthcareai_toolkit import settings

# MedImageParse3D uses the same client interface as 2D MedImageParse.
# The 3D model endpoint accepts base64-encoded NIfTI volumes instead of 2D images.
# Set MIP3D_MODEL_ENDPOINT in your .env file to the 3D endpoint name or resource URI.
endpoint_name = os.environ.get("MIP3D_MODEL_ENDPOINT")
if not endpoint_name:
    raise ValueError(
        "Set MIP3D_MODEL_ENDPOINT in your .env file to the 3D model endpoint name "
        "or full Azure resource URI."
    )
client = MedImageParseClient(endpoint_name=endpoint_name)


In [ ]:
import os

data_root = settings.DATA_ROOT
example_dir = os.path.join(data_root, "segmentation-3d-examples")


In [ ]:
import base64
import json
import matplotlib.pyplot as plt
import numpy as np
import nibabel as nib

from healthcareai_toolkit.data.io import read_nifti


def read_volume_bytes(volume_path):
    """Read a NIfTI volume as raw bytes for base64 encoding."""
    with open(volume_path, "rb") as f:
        return f.read()


def submit_volume(client, volume_path, text_prompt):
    """Submit a NIfTI volume to the MedImageParse3D endpoint.

    Reads the volume file, base64-encodes it, and sends it with the text prompt
    to the model endpoint for volumetric segmentation.
    """
    volume_bytes = read_volume_bytes(volume_path)
    encoded = base64.encodebytes(volume_bytes).decode("utf-8")
    payload = {
        "input_data": {
            "columns": ["image", "text"],
            "index": [0],
            "data": [[encoded, text_prompt]],
        }
    }
    # Submit using the client's underlying payload submission method
    response = client._submit_payload(payload)
    return response


def decode_image_features(json_encoded):
    """Decode a base64-encoded array from the model response."""
    array_metadata = json.loads(json_encoded)
    base64_encoded = array_metadata["data"]
    shape = tuple(array_metadata["shape"])
    dtype = np.dtype(array_metadata["dtype"])
    array_bytes = base64.b64decode(base64_encoded)
    return np.frombuffer(array_bytes, dtype=dtype).reshape(shape)


def show_volume_slice_with_mask(
    volume_slice,
    mask_slice,
    alpha=0.5,
    title=None,
    ax=None,
    colormap="Set1",
):
    """Display a 2D slice from a 3D volume with a segmentation mask overlay."""
    labels = np.unique(mask_slice)
    labels = labels[labels > 0]
    cmap = plt.get_cmap(colormap)

    overlay = np.zeros((*mask_slice.shape, 4), dtype=np.float32)
    if len(labels) > 0:
        norm = plt.Normalize(vmin=labels.min(), vmax=labels.max())
        for label in labels:
            color = cmap(norm(label))[:3]
            overlay[mask_slice == label] = [*color, alpha]

    show = False
    if ax is None:
        ax = plt.gca()
        show = True

    ax.imshow(volume_slice, cmap="gray")
    ax.imshow(overlay)
    ax.axis("off")
    if title is not None:
        ax.set_title(title)
    if show:
        plt.show()


def visualize_3d_segmentation_results(
    volume, results, slice_idx=None, axis=2, colormap="Set1"
):
    """Visualize segmentation results on a central slice of a 3D volume.

    Args:
        volume: 3D numpy array from a NIfTI file.
        results: Model response containing image_features and text_features.
        slice_idx: Slice index to display (default: middle slice along axis).
        axis: Axis along which to extract the slice (0=sagittal, 1=coronal, 2=axial).
        colormap: Matplotlib colormap for mask overlay.
    """
    text_features = results[0]["text_features"]
    masks = results[0]["image_features"]
    if isinstance(masks, str):
        masks = decode_image_features(masks)
    masks = (masks > 0).astype("uint8")

    if slice_idx is None:
        slice_idx = volume.shape[axis] // 2

    vol_slice = np.take(volume, slice_idx, axis=axis)
    # Normalize the slice for display
    vol_slice = ((vol_slice - vol_slice.min()) / (vol_slice.max() - vol_slice.min() + 1e-8) * 255).astype(np.uint8)

    print(f"Volume shape: {volume.shape}")
    print(f"Segmentation mask shape: {masks.shape}")
    print(f"Text features: {text_features}")
    print(f"Displaying slice {slice_idx} along axis {axis}")

    # Handle multiple masks
    if masks.ndim == 4:
        individual_masks = [masks[i] for i in range(masks.shape[0])]
    elif masks.ndim == 3:
        individual_masks = [masks]
    else:
        individual_masks = [masks]

    fig, axes = plt.subplots(1, len(individual_masks) + 1, figsize=(6 * (len(individual_masks) + 1), 5))
    if len(individual_masks) == 0:
        axes = [axes]
    elif len(individual_masks) == 1 and not hasattr(axes, '__len__'):
        axes = [axes, axes]

    # Original slice
    axes[0].imshow(vol_slice, cmap="gray")
    axes[0].set_title(f"Original (slice {slice_idx})")
    axes[0].axis("off")

    for i, mask in enumerate(individual_masks):
        mask_slice = np.take(mask, min(slice_idx, mask.shape[axis] - 1), axis=axis)
        label = text_features[i] if i < len(text_features) else f"Mask {i+1}"
        show_volume_slice_with_mask(vol_slice, mask_slice, title=label, ax=axes[i + 1], colormap=colormap)

    plt.tight_layout()
    plt.show()


def visualize_3d_multi_axis(volume, results, colormap="Set1"):
    """Show segmentation results across axial, coronal, and sagittal views."""
    text_features = results[0]["text_features"]
    masks = results[0]["image_features"]
    if isinstance(masks, str):
        masks = decode_image_features(masks)
    masks = (masks > 0).astype("uint8")

    if masks.ndim == 4:
        # Take the first mask for multi-axis view
        mask = masks[0]
    else:
        mask = masks

    axis_names = ["Sagittal", "Coronal", "Axial"]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax_idx, (ax, name) in enumerate(zip(axes, axis_names)):
        s = volume.shape[ax_idx] // 2
        vol_slice = np.take(volume, s, axis=ax_idx)
        vol_slice = ((vol_slice - vol_slice.min()) / (vol_slice.max() - vol_slice.min() + 1e-8) * 255).astype(np.uint8)
        mask_slice = np.take(mask, min(s, mask.shape[ax_idx] - 1), axis=ax_idx)
        show_volume_slice_with_mask(vol_slice, mask_slice, title=f"{name} (slice {s})", ax=ax, colormap=colormap)
    plt.suptitle(f"Prompt: {' & '.join(text_features)}", fontsize=14)
    plt.tight_layout()
    plt.show()


## 2. 3D Volumetric Segmentation Examples
Below are several examples illustrating MedImageParse3D's prompt-based approach to 3D volumetric segmentation. Unlike the 2D MedImageParse model that processes individual slices, MedImageParse3D accepts full NIfTI volumes and produces 3D segmentation masks that capture spatial context across the entire volume.


### 2.1 Liver Segmentation from Abdominal CT Volume
This example demonstrates volumetric liver segmentation from a full abdominal CT scan. The model processes the entire NIfTI volume and produces a 3D segmentation mask of the liver across all slices.


In [ ]:
volume_path = os.path.join(example_dir, "CT_abdomen_liver.nii.gz")
text_prompt = "liver"

volume = read_nifti(volume_path)
results = submit_volume(client, volume_path, text_prompt)

visualize_3d_segmentation_results(volume, results)


### 2.2 Brain Tumor Segmentation from MRI T1-Gad Volume


This example segments glioblastoma components from a full 3D brain MRI T1-Gadolinium volume. Unlike the 2D MedImageParse which requires extracting a single slice, MedImageParse3D processes the entire volume and produces volumetric masks for the tumor core, enhancing tumor, and non-enhancing tumor regions.


In [ ]:
volume_path = os.path.join(example_dir, "BRATS_T1Gad.nii.gz")
text_prompt = "tumor core & enhancing tumor & non-enhancing tumor"

volume = read_nifti(volume_path)
results = submit_volume(client, volume_path, text_prompt)

visualize_3d_segmentation_results(volume, results)


### 2.3 Brain Tumor Segmentation from MRI-FLAIR Volume


This example uses a full 3D MRI-FLAIR (Fluid-Attenuated Inversion Recovery) volume to segment the whole tumor, tumor core, and surrounding edema. FLAIR sequences are particularly sensitive to lesions and edema, making them ideal for comprehensive tumor characterization.


In [ ]:
volume_path = os.path.join(example_dir, "BRATS_FLAIR.nii.gz")
text_prompt = "whole tumor & tumor core & edema"

volume = read_nifti(volume_path)
results = submit_volume(client, volume_path, text_prompt)

visualize_3d_segmentation_results(volume, results)


### 2.4 Kidney, Tumor, and Cyst Segmentation from Abdominal CT Volume


This example performs volumetric segmentation of renal anatomy from a full abdominal CT volume. The model segments kidneys, tumors, and cysts across the entire 3D volume, providing complete spatial extent of each structure.


In [ ]:
volume_path = os.path.join(example_dir, "CT_kidney.nii.gz")
text_prompt = "kidney & tumor & cyst"

volume = read_nifti(volume_path)
results = submit_volume(client, volume_path, text_prompt)

visualize_3d_segmentation_results(volume, results)


### 2.5 Cardiac Chamber Segmentation from MRI Volume


This example demonstrates cardiac chamber segmentation from a 3D cardiac MRI volume. The model segments the left ventricle and left atrium across the full volume, enabling volumetric assessment of cardiac function and chamber size.


In [ ]:
volume_path = os.path.join(example_dir, "cardiac_MRI.nii.gz")
text_prompt = "left ventricle & left atrium"

volume = read_nifti(volume_path)
results = submit_volume(client, volume_path, text_prompt)

visualize_3d_segmentation_results(volume, results)


### 2.6 Lung Segmentation and Nodule Detection from Chest CT Volume


This example performs volumetric lung segmentation and nodule detection from a full chest CT volume. The 3D model can capture the complete spatial extent of nodules across contiguous slices, unlike slice-by-slice 2D approaches.


In [ ]:
volume_path = os.path.join(example_dir, "CT_chest_lung.nii.gz")
text_prompt = "left lung & right lung & nodule"

volume = read_nifti(volume_path)
results = submit_volume(client, volume_path, text_prompt)

visualize_3d_segmentation_results(volume, results)


### 2.7 Multi-Axis Visualization
The example below shows how to view the same volumetric segmentation result across axial, coronal, and sagittal planes. This provides a comprehensive view of the 3D segmentation quality using the output from the previous query.


In [ ]:
visualize_3d_multi_axis(volume, results)


### 2.8 Liver Tumor Segmentation from Abdominal CT Volume


This example demonstrates segmentation of liver anatomy and tumors from a contrast-enhanced abdominal CT volume. The model identifies both normal liver parenchyma and tumor regions in 3D.


In [ ]:
volume_path = os.path.join(example_dir, "CT_liver_tumor.nii.gz")
text_prompt = "liver & tumor"

volume = read_nifti(volume_path)
results = submit_volume(client, volume_path, text_prompt)

visualize_3d_segmentation_results(volume, results)


### 2.9 Pancreas Segmentation from Abdominal CT Volume


The pancreas is a notoriously difficult organ to segment due to its variable shape and small size. MedImageParse3D leverages 3D spatial context to produce more accurate segmentations than slice-by-slice approaches.


In [ ]:
volume_path = os.path.join(example_dir, "CT_pancreas.nii.gz")
text_prompt = "pancreas"

volume = read_nifti(volume_path)
results = submit_volume(client, volume_path, text_prompt)

visualize_3d_segmentation_results(volume, results)


### 2.10 Spleen Segmentation from Abdominal CT Volume


This example shows spleen segmentation from an abdominal CT volume. The spleen is a relatively well-defined organ that demonstrates the model's ability to produce clean, contiguous 3D masks.


In [ ]:
volume_path = os.path.join(example_dir, "CT_abdomen_spleen.nii.gz")
text_prompt = "spleen"

volume = read_nifti(volume_path)
results = submit_volume(client, volume_path, text_prompt)

visualize_3d_segmentation_results(volume, results)


### 2.11 Multi-Organ Segmentation from Abdominal CT Volume


This example demonstrates the model's ability to simultaneously segment multiple abdominal organs from a single CT volume using a compound text prompt. This showcases the power of prompt-based 3D segmentation for comprehensive anatomical analysis.


In [ ]:
volume_path = os.path.join(example_dir, "CT_abdomen_multi.nii.gz")
text_prompt = "liver & kidney & spleen & pancreas"

volume = read_nifti(volume_path)
results = submit_volume(client, volume_path, text_prompt)

visualize_3d_segmentation_results(volume, results, colormap="Dark2")


### 2.12 Hippocampus Segmentation from Brain MRI Volume


The hippocampus is a small, complex structure in the brain important for memory and learning. Accurate 3D segmentation of the hippocampus from MRI is valuable for studying neurodegenerative diseases like Alzheimer's.


In [ ]:
volume_path = os.path.join(example_dir, "MRI_brain_hippocampus.nii.gz")
text_prompt = "hippocampus"

volume = read_nifti(volume_path)
results = submit_volume(client, volume_path, text_prompt)

visualize_3d_segmentation_results(volume, results)


# 3. References

For further details see:

 - [BiomedParse OSS GitHub project](https://microsoft.github.io/BiomedParse/)

 - [BiomedParse Paper](https://arxiv.org/abs/2405.12971)

 - [BiomedParse 3D: 3D Volumetric Biomedical Image Segmentation](https://arxiv.org/abs/2505.06993)
